# HCC unlabeled rest — UNI2 A3 best_mlp

8 CODEX HCC **rest** regions without a CODEX cell-type CSV. Not the Visium `External_val/HCC` JPEGs. Notebook draws the **5 largest**. No TLS / SRI / TNI.

Checkpoint: `data/PanCancer/result_uni2_A3_adjacent_spatial_k8mean_gate_paired_v2_seed42/cross_dataset_cv/A3_adjacent_spatial_k8mean_gate/best_mlp_gpu.pt` (fold 0).

The UNI notebook `Pred_visual_hcc_unlabel.ipynb` still reads `pred_unlabel/HCC_unlabel`. A2 is `Pred_visual_hcc_unlabel_uni2_A2.ipynb`.

```bash
CUDA_DEVICE=3 bash code/External_val/HCC_unlabel/demo_uni2_A3.sh predict
```


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
os.environ["UNI2_CKPT"] = str(
    REPO
    / "data/PanCancer/result_uni2_A3_adjacent_spatial_k8mean_gate_paired_v2_seed42"
    / "cross_dataset_cv/A3_adjacent_spatial_k8mean_gate/best_mlp_gpu.pt"
)
os.environ["UNI2_PRED_DIRNAME"] = "pred_uni2_A3_bestmlp_paired_v2_seed42"
sys.path.insert(0, str(REPO / "code" / "External_val"))

import unlabeled_external as ue

COHORT = "HCC_unlabel"
EMBEDDER = "uni2"
SAVE_FIGURES = False
FIGSIZE = (17.2, 12.5)
DPI = 160
SAVE_DPI = 300
HE_MAX_SIDE = 1400
MARKER_PX = 1.0

print("checkpoint", ue._ckpt(EMBEDDER))
cov = ue.coverage(COHORT, EMBEDDER)
samples = ue.viz_samples(COHORT, embedder=EMBEDDER)
print(f"{COHORT}: {len(cov)} unlabeled slides; drawing {len(samples)}")
print("viz", samples)
display(cov.loc[cov["sample"].isin(samples)])


In [ ]:
display(Markdown("## Predicted L1 / L12 / L2 (row 1 HE, row 2 points)"))
for sample in samples:
    pred = ue.pred_h5ad(ue._cohort(COHORT, EMBEDDER), sample)
    if not pred.is_file():
        print("missing pred", sample, "— run demo_uni2_A3.sh predict")
        continue
    figs = ue.plot_spatial(
        COHORT,
        [sample],
        embedder=EMBEDDER,
        save=SAVE_FIGURES,
        figsize=FIGSIZE,
        dpi=DPI,
        save_dpi=SAVE_DPI,
        he_max_side=HE_MAX_SIDE,
        marker_px=MARKER_PX,
    )
    for fig in figs:
        display(fig)
        plt.close(fig)


In [ ]:
display(Markdown("## Predicted composition (no GT)"))
have = [s for s in samples if ue.pred_h5ad(ue._cohort(COHORT, EMBEDDER), s).is_file()]
if have:
    figs = ue.plot_composition(
        COHORT, have, embedder=EMBEDDER, save=SAVE_FIGURES, dpi=DPI, save_dpi=SAVE_DPI
    )
    for fig in figs:
        display(fig)
        plt.close(fig)
